# Combining Models: Bagging, Boosting, Stacking (Bishop BP12)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/ensemble_methods.ipynb)

Companion notebook to the [blog post](https://sesen.ai/blog/combining-models-bagging-boosting-stacking). Five strategies for combining models, all in plain Python:

1. **Bagging** — Breiman 1996, bootstrap aggregation
2. **AdaBoost** — Freund & Schapire 1996, from Bishop eqs 14.15-14.18
3. **Gradient Boosting** — Friedman 2001, residual fitting
4. **Stacking** — meta-learning over base predictions
5. **Mixture of Experts** — Jacobs et al. 1991, input-dependent gating

Benchmarked against `sklearn` and `xgboost` on California Housing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import (
    fetch_california_housing, make_classification, make_moons, make_regression,
)
from sklearn.ensemble import (
    AdaBoostClassifier, BaggingRegressor, GradientBoostingRegressor,
    RandomForestRegressor, StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

rng = np.random.default_rng(0)

## 1. AdaBoost from scratch (Bishop eqs 14.15-14.18)

Three steps:
1. Initialise $w_n = 1/N$
2. For $m = 1, \dots, M$: fit stump on weighted data, compute $\epsilon_m$ and $\alpha_m = \ln((1-\epsilon_m)/\epsilon_m)$, update weights
3. Predict $Y_M(\mathbf{x}) = \text{sign}(\sum_m \alpha_m y_m(\mathbf{x}))$

In [ ]:
class AdaBoostScratch:
    def __init__(self, n_rounds=150, base_max_depth=1):
        self.n_rounds = n_rounds
        self.base_max_depth = base_max_depth
    def fit(self, X, t):
        N = X.shape[0]
        w = np.ones(N) / N
        self.alphas_, self.learners_ = [], []
        for m in range(self.n_rounds):
            stump = DecisionTreeClassifier(max_depth=self.base_max_depth, random_state=m)
            stump.fit(X, t, sample_weight=w)
            preds = stump.predict(X)
            eps = float(np.clip(np.sum(w * (preds != t)) / np.sum(w), 1e-12, 1-1e-12))
            alpha = np.log((1 - eps) / eps)
            w = w * np.exp(alpha * (preds != t))
            w = w / w.sum()
            self.alphas_.append(alpha)
            self.learners_.append(stump)
        return self
    def predict(self, X):
        F = sum(a * h.predict(X) for a, h in zip(self.alphas_, self.learners_))
        return np.sign(F).astype(int)

X, y = make_moons(n_samples=300, noise=0.32, random_state=0)
# Sprinkle 5% label noise
flip = rng.random(300) < 0.05
y = np.where(flip, 1 - y, y)
t = 2 * y - 1

clf = AdaBoostScratch(n_rounds=150).fit(X, t)
print(f'AdaBoost (scratch) training accuracy: {(clf.predict(X) == t).mean():.4f}')

# Sanity-check against sklearn SAMME
sk_ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=0),
                            n_estimators=150, algorithm='SAMME', random_state=0).fit(X, y)
print(f'AdaBoost (sklearn) training accuracy: {sk_ada.score(X, y):.4f}')

In [ ]:
# Plot final decision boundary
x_min, x_max = X[:, 0].min()-0.5, X[:, 0].max()+0.5
y_min, y_max = X[:, 1].min()-0.5, X[:, 1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.contourf(xx, yy, Z, levels=[-2, 0, 2], colors=['#fbe4dc', '#dde7f7'], alpha=0.8)
ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=1.5)
ax.scatter(X[t==1, 0], X[t==1, 1], s=22, c='#2d6cdf', edgecolors='white', linewidth=0.5, label='+1')
ax.scatter(X[t==-1, 0], X[t==-1, 1], s=22, c='#cc3344', edgecolors='white', linewidth=0.5, label='-1')
ax.set_aspect('equal', 'box')
ax.set_title('AdaBoost from scratch: 150 stumps')
ax.legend()
plt.show()

## 2. Gradient Boosting from scratch (Friedman 2001)

Each new tree fits the residual $y - F(\mathbf{x})$ of the current cumulative model. Add it back with a learning-rate shrinkage. That's all gradient boosting is for squared-error loss.

In [ ]:
class GradientBoostingScratch:
    def __init__(self, n_rounds=300, learning_rate=0.05, max_depth=3):
        self.n_rounds, self.lr, self.max_depth = n_rounds, learning_rate, max_depth
    def fit(self, X, y):
        self.f0_ = float(np.mean(y))
        F = np.full_like(y, self.f0_, dtype=float)
        self.trees_ = []
        for m in range(self.n_rounds):
            residual = y - F
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=m)
            tree.fit(X, residual)
            F = F + self.lr * tree.predict(X)
            self.trees_.append(tree)
        return self
    def predict(self, X):
        F = np.full(X.shape[0], self.f0_, dtype=float)
        for tree in self.trees_:
            F = F + self.lr * tree.predict(X)
        return F

X_r, y_r = make_regression(n_samples=1000, n_features=10, noise=10.0, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.25, random_state=0)

gb_scratch = GradientBoostingScratch(n_rounds=300, learning_rate=0.05, max_depth=3).fit(X_tr, y_tr)
gb_sklearn = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                         random_state=0).fit(X_tr, y_tr)

mse_scratch = mean_squared_error(y_te, gb_scratch.predict(X_te))
mse_sk = mean_squared_error(y_te, gb_sklearn.predict(X_te))
print(f'Test MSE (scratch): {mse_scratch:.2f}')
print(f'Test MSE (sklearn): {mse_sk:.2f}')

## 3. Bagging from scratch

Bootstrap M times, fit a tree on each bootstrap, average predictions. Watch the variance shrink (slowly, because trees are correlated).

In [ ]:
def bagging_predict(X_train, y_train, X_test, n_estimators=50, seed=0):
    rng = np.random.default_rng(seed)
    N = X_train.shape[0]
    preds = np.zeros((n_estimators, X_test.shape[0]))
    for m in range(n_estimators):
        idx = rng.integers(0, N, size=N)
        tree = DecisionTreeRegressor(random_state=seed + m)
        tree.fit(X_train[idx], y_train[idx])
        preds[m] = tree.predict(X_test)
    return preds.mean(axis=0)

# Variance vs M (uncorrelated baseline = var_1 / M)
def true_fn(x): return np.sin(2*np.pi*x) + 0.3*x
X_test = np.linspace(0, 1, 200).reshape(-1, 1)

Ms, vars_ = [1, 2, 5, 10, 25, 50, 100, 200], []
for M in Ms:
    preds = []
    for rep in range(20):
        rng_in = np.random.default_rng(rep + 100)
        X_train = rng_in.uniform(0, 1, 80).reshape(-1, 1)
        y_train = true_fn(X_train.ravel()) + rng_in.normal(0, 0.3, 80)
        preds.append(bagging_predict(X_train, y_train, X_test, n_estimators=M, seed=rep))
    vars_.append(float(np.array(preds).var(axis=0).mean()))

print(f'Variance @ M=1:   {vars_[0]:.4f}')
print(f'Variance @ M=200: {vars_[-1]:.4f}')
print(f'Ideal 1/M shrinkage @ M=200: {vars_[0]/200:.5f}  (actual is much higher because trees are correlated)')

## 4. Headline benchmark on California Housing

Five strategies on the same train/test split.

> XGBoost requires libomp on macOS (`brew install libomp`) and is pre-installed on Colab.

In [ ]:
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not available; install with `pip install xgboost`')

data = fetch_california_housing()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

models = [
    ('Single tree', DecisionTreeRegressor(random_state=0)),
    ('Bagging (50)', BaggingRegressor(n_estimators=50, random_state=0)),
    ('Random Forest (200)', RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)),
    ('Gradient Boosting (sklearn, 500)',
     GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                               subsample=0.8, random_state=0)),
]
if HAS_XGB:
    models.append(('XGBoost (500)',
                   xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                     random_state=0, n_jobs=-1, tree_method='hist')))

for name, model in models:
    model.fit(X_tr, y_tr)
    rmse = float(np.sqrt(mean_squared_error(y_te, model.predict(X_te))))
    print(f'  {name:35s}  test RMSE = {rmse:.4f}')

## 5. Stacking

Three different base learners + a logistic meta-model trained on their out-of-fold predictions.

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10,
                            n_redundant=5, n_classes=2, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

base_models = [
    ('knn',    KNeighborsClassifier(n_neighbors=15)),
    ('tree',   DecisionTreeClassifier(max_depth=6, random_state=0)),
    ('logreg', LogisticRegression(max_iter=500)),
]
for name, m in base_models:
    m.fit(X_tr, y_tr)
    print(f'  {name:8s} accuracy = {accuracy_score(y_te, m.predict(X_te)):.4f}')

stack = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=500),
    cv=5,
).fit(X_tr, y_tr)
print(f'  STACK    accuracy = {accuracy_score(y_te, stack.predict(X_te)):.4f}')

## 6. Mixture of Experts (toy regression)

Two linear experts plus a sigmoid gating network, fit by EM on piecewise-linear data.

In [ ]:
rng = np.random.default_rng(4)
N = 400
x = rng.uniform(-3, 3, N)
y_true = np.where(x < 0, 1.5*x + 0.2, -1.0*x + 0.2)
y = y_true + rng.normal(0, 0.25, N)

a = np.array([rng.normal() for _ in range(2)])
b = np.array([rng.normal() for _ in range(2)])
sigma = np.array([1.0, 1.0])
g = np.array([0.0, 1.0])

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

for it in range(150):
    pi1 = sigmoid(g[0] + g[1]*x); pi2 = 1 - pi1
    mu1, mu2 = a[0]*x + b[0], a[1]*x + b[1]
    p1 = pi1 * np.exp(-0.5*((y-mu1)/sigma[0])**2) / (sigma[0]*np.sqrt(2*np.pi))
    p2 = pi2 * np.exp(-0.5*((y-mu2)/sigma[1])**2) / (sigma[1]*np.sqrt(2*np.pi))
    gamma1 = p1 / (p1 + p2 + 1e-12); gamma2 = 1 - gamma1
    for k, gamma in enumerate([gamma1, gamma2]):
        X1 = np.c_[x, np.ones_like(x)]
        theta = np.linalg.lstsq(X1 * gamma[:, None], y * gamma, rcond=None)[0]
        a[k], b[k] = theta[0], theta[1]
        resid = y - (a[k]*x + b[k])
        sigma[k] = max(np.sqrt(np.sum(gamma*resid**2) / np.sum(gamma)), 1e-3)
    Xg = np.c_[np.ones_like(x), x]
    for _ in range(3):
        p = sigmoid(Xg @ g); W = np.diag(p*(1-p) + 1e-6)
        z = Xg @ g + (gamma1 - p) / (p*(1-p) + 1e-6)
        g = np.linalg.solve(Xg.T @ W @ Xg, Xg.T @ W @ z)

print(f'Expert slopes:     {a.round(3)}  (true: [-1.0, 1.5])')
print(f'Expert intercepts: {b.round(3)}  (true: [0.2, 0.2])')
print(f'Gating params:     g0={g[0]:.3f}, g1={g[1]:.3f}  (gate transitions near x = -g0/g1 = {-g[0]/g[1]:.3f}, true: 0)')

## Exercises

1. **Boosting overfits**: Run `GradientBoostingScratch` for 5,000 rounds on the small regression dataset above. Plot train and test MSE. Identify the round at which test MSE starts increasing — that's where early stopping should kick in.

2. **Diverse base learners help stacking**: Replace one of the three stacking base models with a copy of another (e.g. use two KNN models). Does stacking still improve over the best individual? Why or why not?

3. **Random Forest closes the bagging gap**: Re-run the variance-vs-M experiment using `RandomForestRegressor(max_features=1)` instead of plain bagging. Does the curve get closer to the ideal $1/M$ baseline?

4. **MoE with more experts**: Generate piecewise data with three slopes (kinks at $x=-1$ and $x=+1$). Extend the MoE EM to $K=3$ experts with a softmax gating network. Verify the gates split into three regions.

5. **XGBoost tuning**: On California Housing, run a small grid over `n_estimators`, `max_depth`, `learning_rate`. What's the best test RMSE you can achieve? How much room is there above the 0.475 baseline?